# 4. Why are there so few delayed unpaired jets?

Emil suggested lowering the momentum cut and looking at all jet times. I try that here and check which runs have cells. The unpaired data are my sample for beam-induced background (BIB). For signal I use simulated jets from long-lived particles (LLPs). [Model comparison](03_train_and_compare.ipynb)

In [ ]:
from collections import Counter
from pathlib import Path
import json

import awkward as ak
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

project = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "DataForEmre").is_dir() and (folder / "notebooks").is_dir()
)
data = project / "DataForEmre"
outputs = project / "analysis/outputs"

## Which runs have cells?

A run is one period of ATLAS data-taking. I match Emil's separate vertex files to the cell files by run and event number, then check the luminosity block. The vertex files store the same event-number bits as a signed integer, so I read them as unsigned. This table counts **events**. One event can have several jets.

In [ ]:
unpaired = data / "non_collision/unpaired"
llp_files = sorted((data / "collision/LLP/N1N2_rpv_1400_10ns").glob("*.root"))
cell_files = sorted((unpaired / "physics_Background_NTuple_unsplit_v2").glob("*.root"))
vertex_files = sorted((unpaired / "user.ezaya.UNPAIRED_nPV_nocells.EXT0").glob("*.root"))
assert llp_files and cell_files and vertex_files, "Check the three local ROOT folders."

In [ ]:
cell_keys = set()
cell_runs = Counter()
cell_id = ["runNumber", "eventNumber", "lumiBlock"]
for path in cell_files:
    with uproot.open(path) as source:
        ids = source["JetContent"].arrays(cell_id, library="np")
    run, event, lumi = (ids[field] for field in cell_id)
    keys = set(zip(map(int, run), map(int, event), map(int, lumi)))
    assert len(keys) == len(run) and cell_keys.isdisjoint(keys), "Repeated cell event."
    cell_keys.update(keys)
    cell_runs.update(map(int, run))

In [ ]:
vertex_id = ["runNumber", "eventNumber", "lumiBlockNumber", "nPrimaryVertices"]
sidecar_runs = Counter()
vertices = {}

def add_vertex_chunk(chunk):
    run, event, lumi, count = (chunk[field] for field in vertex_id)
    runs, counts = np.unique(run, return_counts=True)
    sidecar_runs.update(dict(zip(map(int, runs), map(int, counts))))
    needed = np.isin(run, list(cell_runs))
    assert event.dtype == np.dtype("int32"), "Recheck event ID storage before matching."
    event = event.astype(np.uint32)
    for r, e, block, n in zip(run[needed], event[needed], lumi[needed], count[needed]):
        key = (int(r), int(e), int(block))
        if key in cell_keys:
            assert key not in vertices and int(n) >= 0, f"Repeated match or negative vertex count for {key}"
            vertices[key] = int(n)

In [ ]:
for path in vertex_files:
    with uproot.open(path) as source:
        for chunk in source["BIB"].iterate(vertex_id, step_size=100_000, library="np"):
            add_vertex_chunk(chunk)

runs = sorted(cell_runs.keys() | sidecar_runs.keys())
run_table = pd.DataFrame({
    "Run": runs,
    "Events with cells": [cell_runs[run] for run in runs],
    "Rows in vertex files": [sidecar_runs[run] for run in runs],
}).set_index("Run")
display(run_table)
assert cell_keys == vertices.keys(), "Some local cell events have no valid vertex match."

Only unpaired run **484258** has cells here. **486706** has jet variables in the files without cells. I'd need its cells to test ParT on a second run.

## What happens when I loosen the cuts?

I keep zero-vertex events for unpaired data, matched jets for LLP, and $|\eta|<1.7$ for both. I start at $p_T\geq30$ GeV with energy in both calorimeters ($0<f_{\rm Tile}<1$). Then I lower the threshold to 15 GeV and finally allow the Tile fraction to reach 0 or 1.

I count all jets, delayed jets ($t_{\rm ATLAS}>3$ ns) and the subset arriving by 10 ns.

In [ ]:
jet_fields = ["jet_pt", "jet_eta", "jet_time_ATLAS", "jet_TileEnergyFrac"]
jet_selections = [(30, True), (15, True), (15, False)]
selection_names = ["30 GeV, mixed", "15 GeV, mixed", "15 GeV, Tile fraction 0 to 1"]
region_names = ["All times", "Time > 3 ns", "3 < time ≤ 10 ns"]

def valid(values):
    return np.isfinite(values) & (values != -999999)

def zero_vertex_events(rows):
    run, event, lumi = (ak.to_numpy(rows[field]) for field in cell_id)
    keys = zip(run, event, lumi)
    return ak.Array([vertices[key] == 0 for key in keys])

In [ ]:
def jet_masks(rows, sample):
    pt, eta, time, tile = (rows[field] for field in jet_fields)
    valid_jet = valid(pt) & valid(eta) & valid(time) & (abs(eta) < 1.7)
    if sample == "LLP":
        valid_jet = valid_jet & (rows["jet_is_llp_matched"] == 1)
    else:
        valid_jet = valid_jet & zero_vertex_events(rows)
    mixed = (tile > 0) & (tile < 1)
    inclusive = (tile >= 0) & (tile <= 1)
    for min_pt, require_mixed in jet_selections:
        calorimeters = mixed if require_mixed else inclusive
        keep = valid_jet & (pt >= 1000 * min_pt) & calorimeters
        delayed = keep & (time > 3)
        yield keep, delayed, delayed & (time <= 10)

In [ ]:
def count_sample(paths, sample):
    fields = jet_fields + (cell_id if sample == "unpaired" else ["jet_is_llp_matched"])
    counts = np.zeros((len(jet_selections), len(region_names)), dtype=np.int64)
    for path in paths:
        with uproot.open(path) as source:
            batches = source["JetContent"].iterate(fields, step_size=10_000, library="ak")
            for rows in batches:
                for i, selections in enumerate(jet_masks(rows, sample)):
                    counts[i] += [int(ak.sum(keep)) for keep in selections]
    return counts

In [ ]:
selection_counts = {}
for sample, paths in [("LLP", llp_files), ("unpaired", cell_files)]:
    counts = count_sample(paths, sample)
    table = pd.DataFrame(counts, index=selection_names, columns=region_names)
    table["Delayed (%)"] = (100 * table["Time > 3 ns"] / table["All times"]).round(2)
    assert (table["All times"] >= table["Time > 3 ns"]).all()
    assert (table["Time > 3 ns"] >= table["3 < time ≤ 10 ns"]).all()
    assert (np.diff(counts, axis=0) >= 0).all()
    print(sample)
    display(table)
    selection_counts[sample] = table

Lowering the momentum cut takes me from **240 to 326** delayed unpaired jets before the train/test split. Allowing all or none of the energy in Tile takes me to **11,889**. That's a much bigger change, and it includes jets outside the training selection.

## Two checks from earlier runs

These tables use local results in `analysis/outputs/`. First I compare models using whole-jet variables across runs. Each cut kept 95% of delayed LLP jets on validation. The LLP test sample stays the same across runs.

In [ ]:
def saved_file(study, name):
    path = outputs / study / name
    if path.is_file():
        return path
    print(f"Local report unavailable: {path.relative_to(project)}")
    return None

def ratio(success, total):
    return f"{int(success):,} / {int(total):,}"

In [ ]:
path = saved_file("unpaired_run_study", "results.csv")
if path is not None:
    report = pd.read_csv(path)
    delayed = report.query("region == 'delayed' and validation_region == 'delayed'").copy()
    delayed["Unpaired tagged / available"] = [ratio(hit, total) for hit, total in
        zip(delayed["unpaired_tagged"], delayed["unpaired_test_jets"])]
    labels = {"time_only": "Time only", "kinematics_only": "Kinematics only",
              "time_and_kinematics": "Time and kinematics"}
    display(delayed.pivot(index="run", columns="model", values="Unpaired tagged / available")
            .rename(columns=labels).reindex(columns=labels.values()))
    llp = delayed[["model", "llp_kept", "llp_test_jets"]].drop_duplicates().set_index("model")
    print("Delayed LLP test jets, shared across runs")
    display(llp.rename(index=labels, columns={"llp_kept": "LLP kept", "llp_test_jets": "LLP tested"}))

Does ParT use the position of a hit? I shuffle times between cells of the same jet, keeping the trained model and cut fixed. The jet has the same list of times, assigned to different positions.

I also try $-N_{\rm cells}$ alone, with its own validation cut. Fewer cells then means a higher BIB score.

In [ ]:
path = saved_file("cell_input_study", "metrics.json")
if path is not None:
    report = json.loads(path.read_text())
    print("Median test cell counts", report["median_cell_count_test"])
    rows = []
    for method, stats in report["comparisons"]["delayed"].items():
        hits, totals = stats["confusion"], stats["class_counts"]
        rows.append({"Method": method.replace("_", " "),
                     "Unpaired tagged / total": ratio(hits["BIB_tagged_BIB"], totals["BIB"]),
                     "LLP kept / total": ratio(hits["LLP_kept_as_LLP"], totals["LLP"])})
    display(pd.DataFrame(rows).set_index("Method"))

Cell count alone tagged **41/42** delayed unpaired jets and kept **715/816** LLP jets. ParT kept **789/816** LLP jets before shuffling and fewer after each shuffle. That suggests it uses the link between time and position. I'd check this on another run.